Selected Column, Merge each year, Delete Duplicate

In [ ]:
import pandas as pd
import ast
import os

DATA_DIR = "./data"
OUTPUT_DIR = "./outputs"

In [ ]:
# Set the year being processed in this run (repeat this notebook for each year: 2021-2025)
YEAR = "2025"
folder_path = os.path.join(DATA_DIR, "Dataset", YEAR)

In [3]:
columns_needed = [
    "tweet_id",

    "core.user_results.result.core.name",
    "core.user_results.result.core.screen_name",
    "core.user_results.result.legacy.followers_count",

    "legacy.full_text",
    "legacy.created_at",
    "legacy.lang",

    "core.user_results.result.location.location",
    "legacy.place.full_name",

    "legacy.favorite_count",
    "legacy.retweet_count",
    "legacy.reply_count",

    "legacy.entities.user_mentions",
    "legacy.in_reply_to_screen_name",
]

In [4]:
#extract mention
def extract_mentions(mention_data):
    if mention_data is None:
        return None
    
    try:
        if isinstance(mention_data, str):
            mention_data = ast.literal_eval(mention_data)

        if isinstance(mention_data, list):
            return [m.get("screen_name") for m in mention_data if isinstance(m, dict)]

    except:
        return None
    
    return None

In [5]:
all_data = []

for file in os.listdir(folder_path):

    if file.endswith(".csv"):

        print("Processing:", file)

        df = pd.read_csv(os.path.join(folder_path, file))

        # expand raw JSON
        raw_dict = df["raw"].apply(ast.literal_eval)
        raw_df = pd.json_normalize(raw_dict)

        df_final = pd.concat([df.drop(columns=["raw"]), raw_df], axis=1)

        # ===============================
        # ambil kolom yang tersedia saja
        # ===============================
        df_selected = df_final[[c for c in columns_needed if c in df_final.columns]]

        # kalau ada kolom yang tidak muncul, buat kolom kosong
        for col in columns_needed:
            if col not in df_selected.columns:
                df_selected[col] = None

        # ===============================
        # rename kolom
        # ===============================
        df_selected = df_selected.rename(columns={
            "core.user_results.result.core.name": "display_name",
            "core.user_results.result.core.screen_name": "username",
            "core.user_results.result.legacy.followers_count": "followers",
            "legacy.full_text": "tweet",
            "legacy.created_at": "created_at",
            "legacy.lang": "lang",
            "legacy.favorite_count": "likes",
            "legacy.retweet_count": "retweets",
            "legacy.reply_count": "replies",
            "legacy.entities.user_mentions": "mentions",
            "legacy.in_reply_to_screen_name": "reply_to_username",
            "core.user_results.result.location.location": "user_location",
            "legacy.place.full_name": "tweet_location"
        })

        # ===============================
        # clean mentions
        # ===============================
        df_selected["mentions"] = df_selected["mentions"].apply(extract_mentions)

        df_selected["mentions"] = df_selected["mentions"].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else ""
        )

        # username konsisten
        df_selected["username"] = df_selected["username"].str.lower()
        df_selected["reply_to_username"] = df_selected["reply_to_username"].str.lower()

        all_data.append(df_selected)

Processing: Key1_Apr_25.csv
Processing: Key1_Jan_25.csv
Processing: Key1_Jan_Mar_25.csv
Processing: Key1_Jul_25.csv


C:\Users\HP\AppData\Local\Temp\ipykernel_25760\372729010.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = None


Processing: Key1_Oct_Des_25.csv
Processing: Key2_Jul_Sep_25.csv


C:\Users\HP\AppData\Local\Temp\ipykernel_25760\372729010.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = None
C:\Users\HP\AppData\Local\Temp\ipykernel_25760\372729010.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = None
C:\Users\HP\AppData\Local\Temp\ipykernel_25760\372729010.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

Processing: Key2_Jun_25.csv
Processing: Key2_Mar_25.csv
Processing: Key3_Apr_Jun_25.csv
Processing: Key3_Jul_Sep_25.csv
Processing: Key3_Mar_25.csv
Processing: Key3_Oct_25.csv


C:\Users\HP\AppData\Local\Temp\ipykernel_25760\372729010.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = None
C:\Users\HP\AppData\Local\Temp\ipykernel_25760\372729010.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = None


In [6]:
# gabungkan semua dataset
df_2025 = pd.concat(all_data, ignore_index=True)

In [7]:
#delete duplicate tweet
df_2025 = df_2025.drop_duplicates(subset="tweet_id")
df_2025 = df_2025.reset_index(drop=True)

In [8]:
# check result
print("Total data:", df_2025.shape)

Total data: (1379, 14)


In [ ]:
# cave to Excel
output_path = os.path.join(OUTPUT_DIR, f"twitter_clean_{YEAR}.xlsx")

df_2025.to_excel(output_path, index=False)

print("File has been saved:", output_path)

File berhasil disimpan: D:\ribkaadevina\college\8.S2\3. SEM 3\Thesis\Playground\Cleansed Dataset\twitter_clean_2025.xlsx


Consolidate into 1 dataset

In [ ]:
import pandas as pd
import os

folder_path = OUTPUT_DIR

all_data = []

for file in os.listdir(folder_path):

    if file.endswith(".xlsx"):

        print("Reading:", file)

        df = pd.read_excel(os.path.join(folder_path, file))

        all_data.append(df)

# gabungkan semua dataframe
df_all = pd.concat(all_data, ignore_index=True)

print("Total data:", df_all.shape)

Reading: dataset_all_years.xlsx
Reading: dataset_clean_text.xlsx
Reading: dataset_ind_only.xlsx
Reading: dataset_no_missing_values.xlsx
Reading: twitter_clean_2021.xlsx
Reading: twitter_clean_2022.xlsx
Reading: twitter_clean_2023.xlsx
Reading: twitter_clean_2024.xlsx
Reading: twitter_clean_2025.xlsx
Total data: (41180, 15)


In [111]:
df_all = df_all.drop_duplicates(subset="tweet_id")

df_all = df_all.reset_index(drop=True)
print("Total data after delete duplicate:", df_all.shape)

Total data after delete duplicate: (8282, 14)


In [ ]:
output_path = os.path.join(OUTPUT_DIR, "dataset_all_years.xlsx")

df_all.to_excel(output_path, index=False)

print("Dataset has been saved:", output_path)

Dataset berhasil disimpan: D:\ribkaadevina\college\8.S2\3. SEM 3\Thesis\Playground\dataset_all_years.xlsx


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
path = os.path.join(OUTPUT_DIR, "dataset_all_years.xlsx")
df = pd.read_excel(path)

df.head()

,tweet_id,display_name,username,followers,tweet,created_at,lang,user_location,tweet_location,likes,retweets,replies,mentions,reply_to_username
0,1406970754688717056,Яizal do,afrkml,350301.0,Gejala yg ditimbulkan memang umum dan sudah di...,Mon Jun 21 13:42:22 +0000 2021,in,Dukung kami di sini →,NaN,245.0,44.0,4.0,NaN,afrkml
1,1404644500203471104,DEMA FUSA UIN RIL,demaafusauinril,30.0,[SELAMAT HARI DEMAM BERDARAH DENGUE ASEAN 2021...,Tue Jun 15 03:38:40 +0000 2021,in,NaN,NaN,2.0,0.0,0.0,NaN,NaN
2,1409518162362568960,🍒,claudiaars__,230.0,Until this minute masih ga habis pikir bisa2ny...,Mon Jun 28 14:24:52 +0000 2021,in,Jakarta,NaN,0.0,0.0,1.0,NaN,NaN
3,1406454852889107968,Fer,tabrakajalahh,28.0,Bisa2nya org kena covid pamer di sosmed. Coba ...,Sun Jun 20 03:32:22 +0000 2021,in,NaN,NaN,0.0,0.0,0.0,NaN,NaN
4,1404615335433563904,JMKI Jawa Tengah,jmkijateng,10.0,Organisasi Kesehatan Dunia ( WHO ) menuliskan ...,Tue Jun 15 01:42:47 +0000 2021,in,NaN,NaN,0.0,0.0,1.0,NaN,jmkijateng


Check Missing Values

In [113]:
missing_values = df_all.isna().sum().sort_values(ascending=False)

print(missing_values)

tweet_location       8156
mentions             6707
reply_to_username    6047
user_location        2580
followers              70
display_name           69
username               69
tweet                  68
created_at             68
lang                   68
likes                  68
retweets               68
replies                68
tweet_id                0
dtype: int64


In [114]:
df_clean = df_all.dropna(subset=[
    "username",
    "tweet",
    "created_at"
])

In [115]:
print(df_clean.shape)
df_clean.isna().sum()

(8213, 14)


tweet_id                0
display_name            0
username                0
followers               1
tweet                   0
created_at              0
lang                    0
user_location        2511
tweet_location       8087
likes                   0
retweets                0
replies                 0
mentions             6638
reply_to_username    5978
dtype: int64

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "dataset_no_missing_values.xlsx")

df_clean.to_excel(output_path, index=False)

print("Dataset has been saved:")
print(output_path)

Dataset berhasil disimpan di:
D:\ribkaadevina\college\8.S2\3. SEM 3\Thesis\Playground\Cleansed Dataset\dataset_no_missing_values.xlsx


Text Cleaning

In [117]:
import re

In [118]:
def clean_tweet(text):

    if not isinstance(text, str):
        return ""

    # lowercase
    text = text.lower()

    # hapus URL
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # hapus mention (@username)
    text = re.sub(r"@\w+", "", text)

    # hapus hashtag (#topic)
    text = re.sub(r"#\w+", "", text)

    # hapus angka
    text = re.sub(r"\d+", "", text)

    # hapus tanda baca
    text = re.sub(r"[^\w\s]", "", text)

    # hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [119]:
df_clean["tweet_clean"] = df_clean["tweet"].apply(clean_tweet)

C:\Users\HP\AppData\Local\Temp\ipykernel_30096\1214264185.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["tweet_clean"] = df_clean["tweet"].apply(clean_tweet)


In [120]:
df_clean = df_clean[df_clean["tweet_clean"].str.strip() != ""]

In [121]:
df_clean[["tweet","tweet_clean"]].head(10)

,tweet,tweet_clean
0,Gejala yg ditimbulkan memang umum dan sudah di...,gejala yg ditimbulkan memang umum dan sudah di...
1,[SELAMAT HARI DEMAM BERDARAH DENGUE ASEAN 2021...,selamat hari demam berdarah dengue asean demam...
2,Until this minute masih ga habis pikir bisa2ny...,until this minute masih ga habis pikir bisanya...
3,Bisa2nya org kena covid pamer di sosmed. Coba ...,bisanya org kena covid pamer di sosmed coba ka...
4,Organisasi Kesehatan Dunia ( WHO ) menuliskan ...,organisasi kesehatan dunia who menuliskan bahw...
5,Hari jum'at 24 juni. Guru SMP ku meninggal di ...,hari jumat juni guru smp ku meninggal di rs kr...
6,"Kasyan tuh penyakit DBD,jantung,kolestrol,dll....",kasyan tuh penyakit dbdjantungkolestroldll sem...
7,Singapura sudah akan beranjak dari Pandemi men...,singapura sudah akan beranjak dari pandemi men...
8,@CNNIndonesia @PutraWadapi Kondisi jakarta yg ...,kondisi jakarta yg mendung dan hujan menambah ...
9,@prdnandre @_chimonica @xxarcturus @txtdrjkt G...,gabisa biasa aja sih karena covid ini bkn peny...


In [ ]:
output_path = os.path.join(OUTPUT_DIR, "dataset_clean_text.xlsx")

df_clean.to_excel(output_path, index=False)

Filter Indonesian Language

In [ ]:
import pandas as pd

file_path = os.path.join(OUTPUT_DIR, "dataset_clean_text.xlsx")

df_clean = pd.read_excel(file_path)

print("Dataset berhasil dibaca")
print("Jumlah data awal:", df_clean.shape[0])

Dataset berhasil dibaca
Jumlah data awal: 8213


In [131]:
print("\nDistribusi bahasa:")
print(df_clean["lang"].value_counts())


Distribusi bahasa:
lang
in    8190
tl       6
tr       3
en       3
et       2
pt       2
ht       2
lv       1
nl       1
eu       1
is       1
it       1
Name: count, dtype: int64


In [132]:
df_clean = df_clean[df_clean["lang"] == "in"]

In [ ]:
after_lang_filter = df_clean.shape[0]

print("Total data after language filter:", after_lang_filter)

Jumlah data setelah filter bahasa: 8190


In [134]:
df_clean = df_clean.reset_index(drop=True)

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "dataset_ind_only.xlsx")

df_clean.to_excel(output_path, index=False)

print("\nDataset has been saved:")
print(output_path)


Dataset hasil filter bahasa disimpan di:
D:\ribkaadevina\college\8.S2\3. SEM 3\Thesis\Playground\Cleansed Dataset\dataset_ind_only.xlsx
